# 6. LLM Risk Analysis

*Goal: Use LLMs to automatically assess the risk level of our identified clusters.*

## Step 1 — Import Libraries

Import the required modules:
- `pandas` for our DataFrame operations. (I like to import with `import pandas as pd` as it a fairly common pattern)
- `SecretStr` from `pydantic`, which is a utility that prevents API keys from accidentally being printed to the console or saved in notebook output.
- `HumanMessage` and `SystemMessage` from `langchain_core.messages` as a means to pass messages to our LLM clients.
- `BaseModel` and `Field` from `pydantic` to define a typed, validated schema for the LLM's structured output

In [2]:
import pandas as pd  # <--- DataFrame usage for dataset operations
from pydantic import SecretStr  # <--- Masked string
from langchain_core.messages import HumanMessage, SystemMessage  # <--- Message types for structuring prompts to the LLM
from pydantic import BaseModel, Field  # <--- For structured data models and validation

## Step 2 — Load the Dataset

Read the Parquet file produced by notebook **4-cluster-data** into a Pandas DataFrame using the `pandas.read_parquet()` function. Name the variable `dataframe`.

> `_04_clustered_commands.parquet` contains the `CommandLine`, `Embedding`, `Cluser_DBSCAN`, and `Cluster_KMeans` columns. Only DBSCAN cluster labels are used in this notebook.

Related Docs:
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_parquet.html#pandas.read_parquet

In [2]:
dataframe = pd.read_parquet("_04_clustered_commands.parquet")

## Step 3 — Deduplicate to Unique Commands

We need to make a unique mapping of commands so we will use the DataFrame function `drop_duplicates(subset="CommandLine")` to produce one row per unique command, then reset the index (do this by calling the `.reset_index(drop=True)` function). Assign the new unique DataFrame to a variabled called `uninque_df`.

> Risk analysis is performed per cluster of unique commands — not per raw event. Deduplicating here avoids sending the same command multiple times in the prompt.

Related Docs:
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop_duplicates.html
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reset_index.html

In [3]:
# Drop duplicate CommandLines so we have a unique list of CommandLines
# and their respected embeddings/vectors
unique_df = dataframe.drop_duplicates(subset="CommandLine")\
    .reset_index(drop=True)

## Step 4 — Define the Prompts

Set three key variables used for every LLM call:

- **`command_sample_size`** — an integer that represents the maximum number of commands sampled from each cluster to keep prompts concise and within token limits. I recommend setting 10.
- **`system_prompt`** — establishes the LLM's role as a DFIR analyst familiar with clustering, and instructs it to keep cluster descriptions short.
- **`user_prompt_template`** — a per-cluster template that injects the cluster number and the sampled commands. **This prompt template must include to format strings for `{cluster_number}` and `{command}` as those will be passed in via a `.format(cluster_naumer=n, command=str)` call on a per cluster basis.

> Using XML-style tags around each command helps the model clearly distinguish individual commands from surrounding text.


Here is an example system prompt:
```
You are a digital forensics and incident response analyst reviewing commands executed on a Windows system. 
You are also proficent with data science and understand machine learning strategies. 
You are using a clusterning algorithm to group commands executed. For each cluster, you are assessing the risk of the commands used in that cluster.
Commands are a sample of the given cluster. Keep the description of the cluster short
and concise.
The command will be between the following tags:  <command> and </command>
```

Here is an example user prompt template:
```
Analyze the following commands given this sample of cluster {cluster_number}. risk_score must be a value between 0-10. Response\n\n<commands to analyze>\n{command}\n</commands to analyze>
```

In [4]:
# The number of samples from each cluster to analyze
command_sample_size = 10

# System prompt for each OpenAI chat request
system_prompt = r"""You are a digital forensics and incident response analyst reviewing commands executed on a Windows system. 
You are also proficent with data science and understand machine learning strategies. 
You are using the DBSCAN clusterning algorithm to group commands executed. For each cluster, assess the risk of the commands used.
Commands are a sample of the given cluster. Keep the description of the cluster short
and concise.
The command will be between the following tags:  <command> and </command>"""

# Chat prompt for each cluster
user_prompt_template = (
    "Analyze the following commands given this sample of cluster {cluster_number}. "
    "risk_score must be a value between 0-10. Response\n\n"
    "<commands to analyze>\n{command}\n</commands to analyze>"
)

## Step 5 — Define the Structured Output Schema
We need to define a data structure that represents the information we want the LLM to give us. One handy feature of newer llm models is the ability to produce structured output. This is easier than trying to use regex patterns to extract data out of LLM response text.

Create a Pydantic `BaseModel` model called `RiskAnalysis` that describes the exact JSON structure the LLM must return:

| Field | Type | Description |
|---|---|---|
| `cluster_description` | `str` | A concise description of what the cluster of commands represents |
| `risk_score` | `int` | A risk score from 0 (benign) to 10 (critical) |

Here is an example of creating a BaseModel class:

```python
class Example(BaseModel):
    """Example class structure."""
    field_1: str = Field(description="A description")
    field_2: int = Field(description="A score")
```

In [3]:
class RiskAnalysis(BaseModel):
    """A risk analysis with details."""
    cluster_description: str = Field(description="The description of the command cluster")
    risk_score: int = Field(description="The score of the risk analysis")

## Step 6 — Initialize the LLM Client
Initialize a Chat client using one of the following methods and assign it to a `chat_model` variable. The nice thing about using LangChain is that it makes it trivial to adjust AI providers. Depending on the provider you are using, import the appropriate Chat class.


- OpenAI `from langchain_openai import ChatOpenAI`
- Google AI Studio `from langchain_google_genai import ChatGoogleGenerativeAI`
- Ollama `from langchain_ollama import ChatOllama`

Both OpenAI and Google will require API keys, but if you are running Ollama locally, you will not need an API key. Here are examples of creating a chat client for these providers. 

```python
ChatOpenAI(
    model="gpt-4o-mini",
    api_key=SecretStr(api_key),
)
```

```python
ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    api_key=SecretStr(api_key),
)
```

```python
ChatOllama(
    model="gemma2",
)
```

Related Docs:
- https://docs.langchain.com/oss/python/integrations/chat/openai
- https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
- https://docs.langchain.com/oss/python/integrations/chat/ollama


If you are using a service that requires an API key, set an api_key variable with something like:

```python
api_key = SecretStr(input("Enter your API key: "))
```

In [ ]:
from langchain_openai import ChatOpenAI  # <--- OpenAI API client for language model interactions

api_key = input("Enter your Azure OpenAI API key: ")

chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=SecretStr(api_key),
)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI  # <--- Google API client for language model interactions

api_key = input("Enter your Google AI Studio API key: ")

chat_model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    api_key=SecretStr(api_key),
)

In [ ]:
from langchain_ollama import ChatOllama  # <--- Ollama API client for language model interactions

chat_model = ChatOllama(
    model="gemma2",
)

###  Configure Structured Output

Set a `structured_llm` variable to a wrapped chat model with LangChain's structured output support so every LLM response is validated against the `RiskAnalysis` schema. To do this, call the `.with_structured_output` function passing in the `RiskAnalysis` class definition and set the `method="json_schema"`.

> This helps ensure each response includes a concise `cluster_description` and an integer `risk_score` that can be consumed directly in later steps.

Example:
```python
chat_model.with_structured_output(
    RiskAnalysis,
    method="json_schema"
)
```

Related Docs:
- https://docs.langchain.com/oss/python/langchain/structured-output

In [ ]:
structured_llm = chat_model.with_structured_output(
    RiskAnalysis,
    method="json_schema"
)

## Step 7 — A Defined Risk Collection Function
Define `collect_risk_summaries_of_unique_commands()` — an `async` function that iterates every DBSCAN cluster and sends a batched prompt to the LLM for each one. We have predefined this function for you since it is a bit more complicated and exact.

For each cluster it:
1. Skips cluster `-1`  and assigns a placeholder `RiskAnalysis` with score 0
3. Samples up to `command_sample_size` commands to keep the prompt concise
4. Wraps each command in `<command>` tags and formats the `user_prompt_template`
5. Calls `structured_llm.ainvoke()` with the system and user messages and appends the typed `RiskAnalysis` response

Related Docs:
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sample.html


In [ ]:
async def collect_risk_summaries_of_unique_commands(
        structured_llm,
        unique_commands_dataframe: pd.DataFrame,
        command_sample_size: int,
        system_prompt: str,
        user_prompt_template: str
) -> tuple[list[int], list[RiskAnalysis]]:
    """Collect risk summaries for each unique command cluster in the provided DataFrame.
    Each cluster is analyzed and a risk summary is generated based on a sample of commands from that cluster.

    Args:
        structured_llm: An instance of a chat model defined with `.with_structured_output()` used to analyze the commands.
        unique_commands_dataframe: A DataFrame containing unique commands and their cluster assignments.
        command_sample_size: The number of commands to sample from each cluster for analysis.
        system_prompt: The system prompt to use for the LLM.
        user_prompt_template: A template for the user prompt, which will be formatted with the cluster number and sampled commands.

    Returns:
        A tuple containing a list of cluster numbers and a list of RiskAnalysis objects for each cluster.

    """
    # Rank the clusters by severity
    _responses: list[RiskAnalysis] = []

    _unique_clusters = unique_commands_dataframe["Cluster_DBSCAN"].unique()
    
    # Iterate each cluster of commands
    for _cluster_number in _unique_clusters:
        if _cluster_number == -1:
            # Skip unclustered data for now
            _responses.append(
                RiskAnalysis(
                    cluster_description="Unclustered commands",
                    risk_score=0
                )
            )
            continue
        
        # Fetch the commands for just the current cluster
        _this_cluster = unique_commands_dataframe[unique_commands_dataframe["Cluster_DBSCAN"] == _cluster_number]

        # Grab a sample of the cluster (or all if less than sample size)
        _this_cluster_sample = _this_cluster if _this_cluster.shape[0] <= command_sample_size \
            else _this_cluster.sample(command_sample_size)

        # Create a string of all the sample commands to insert into the user prompt
        _cmd_list = ["\n".join([f"<command>", cmd, f"</command>"]) for cmd in _this_cluster_sample["CommandLine"]]
        _cmd_body = "\n\n".join(_cmd_list)

        # Craft the user prompt for this cluster of commands
        _user_prompt = user_prompt_template.format(
            cluster_number=_cluster_number,
            command=_cmd_body
        )

        # Get the response and added it to the responses to be returned
        _response: RiskAnalysis = await structured_llm.ainvoke([
            SystemMessage(content=system_prompt),  # <--- System prompt
            HumanMessage(content=_user_prompt)  # <--- User prompt
        ])

        # Append the response to the list of responses
        _responses.append(_response)

    return _unique_clusters, _responses

## Step 8 — Run the Risk Analysis

`await` the `collect_risk_summaries()` coroutine to send all cluster prompts to the LLM and gather the results into `risk_summary_responses`.

> This is the most time-consuming step — one LLM call is made per DBSCAN cluster. The list returned contains one `RiskAnalysis` object per cluster, in the same order as `sorted_culsters`.

In [ ]:
# Collect the risk responses from the llm
unique_clusters, risk_summaries = await collect_risk_summaries_of_unique_commands(
    structured_llm,
    unique_df,
    command_sample_size,
    system_prompt,
    user_prompt_template
)

## Step 9 — Build a Cluster-to-RiskAnalysis Mapping

`zip` the sorted cluster labels with the LLM responses to create a `cluster_mapping` dictionary, where each cluster number maps to its corresponding `RiskAnalysis` object.

> This dictionary is used in the next step to look up each row's description and risk score in a single vectorised `.map()` call.

In [ ]:
cluster_mapping = dict(zip(unique_clusters, risk_summaries))

## Step 10 — Enrich `unique_df` with LLM Risk Outputs

Attach the LLM results back onto each command row by mapping the `Cluster_DBSCAN` column (`unique_df["Cluster_DBSCAN"]`) through `cluster_mapping` (created in Step 9).

Create two new columns:

- **`DBSCANClusterDescription`**: short cluster summary from `RiskAnalysis.cluster_description`
- **`DBSCANClusterRiskScore`**: numeric severity from `RiskAnalysis.risk_score` (0–10)

Using `.map()` keeps the operation concise and column-wise, so every row in `unique_df` inherits the description and score for its cluster label.

Example of creating a new series (column) using map:
`unique_df["DBSCANClusterDescription"] = unique_df["Cluster_DBSCAN"].map(lambda x: ...)`

In [ ]:
unique_df["DBSCANClusterDescription"] = unique_df["Cluster_DBSCAN"].map(lambda x: cluster_mapping[x].cluster_description)
unique_df["DBSCANClusterRiskScore"] = unique_df["Cluster_DBSCAN"].map(lambda x: cluster_mapping[x].risk_score)

## Step 11 — Save the Enriched Dataset

Persist `unique_df` — now containing cluster descriptions and risk scores — to `_06_unique_clustered_commands_with_risk.parquet` as it will be used for lab 8. You can also output this dataframe to a csv/xlsx file for easily viewing the the risk scores and descriptions.

Related Docs:
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_parquet.html
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_csv.html
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.to_excel.html

In [16]:
unique_df.to_parquet("_06_unique_clustered_commands_with_risk.parquet", index=False)
unique_df.to_csv("_06_unique_clustered_commands_with_risk.csv", index=False)